# 01 · Esplorazione su un frame singolo

Domanda: come separare il docente dalla lavagna? Strade confrontate sullo stesso frame:

1. **DeepLab V3 sul frame intero** (quello che fa la demo web di MediaPipe)
2. **DeepLab V3 su un ritaglio** attorno al docente
3. **PoseLandmarker** (rileva la persona, ritaglia, segmenta; dà anche i landmark, quindi il punto indicato)
4. **Geometria**: quadrilatero della lavagna, pixel di persona ∩ lavagna
5. **Colore**: quanto sono separabili pelle, ardesia e gesso

Il frame è quello censurato (docente sostituita da aree a colore medio): i numeri sulla persona vanno riconfermati su un frame reale.

In [ ]:
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision

from image_segmentation.board import detect_board, board_mask, rectify_homography
from image_segmentation.person import PersonSegmenter, MODELS_DIR

ROOT = Path.cwd().parent
VIDEO_CROP = (0, 68, 1674, 783)  # x, y, w, h: area del video nello screenshot del player
x, y, w, h = VIDEO_CROP
frame = cv2.imread(str(ROOT / "data/frame/lezione_prova_cit_censured.png"))[y:y+h, x:x+w]

def show(*imgs, titles=None, size=6):
    fig, axs = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size * 0.6))
    for i, (ax, im) in enumerate(zip(np.atleast_1d(axs), imgs)):
        ax.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
        ax.set_title(titles[i] if titles else ""); ax.axis("off")
    plt.tight_layout(); plt.show()

show(frame, titles=["frame (area video)"], size=10)

## 1–2 · DeepLab V3: frame intero contro ritaglio

In [ ]:
opts = vision.ImageSegmenterOptions(
    base_options=mpt.BaseOptions(model_asset_path=str(MODELS_DIR / "deeplab_v3.tflite")),
    output_confidence_masks=True)
deeplab = vision.ImageSegmenter.create_from_options(opts)
person_conf = lambda img: deeplab.segment(
    mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(img[..., ::-1]))
).confidence_masks[15].numpy_view().squeeze().copy()  # copia: la vista muore con il risultato

full = person_conf(frame)
crop = frame[292:712, 330:760]
cropped = person_conf(crop)
print(f"frame intero: confidenza max {full.max():.3f}, pixel > 0.5: {(full > 0.5).sum()}")
print(f"ritaglio:     confidenza max {cropped.max():.3f}, pixel > 0.5: {(cropped > 0.5).sum()}")
show(full, crop, cropped, titles=["persona, frame intero", "ritaglio", "persona, ritaglio"])

Il modello gira a 257×257: sul frame intero il docente diventa una ventina di pixel e sparisce.
Il problema è la **risoluzione**, non il colore. La media dei colori per regione aiutava nella demo
perché rendeva le parti più uniformi, ma richiede di conoscere già la maschera: è circolare.

## 3 · PoseLandmarker: maschera e punto indicato

In [ ]:
with PersonSegmenter() as segment:
    person = segment(frame)
print("punta dell'indice:", person.pointer, " angolo braccio-busto:", round(person.arm_angle), "gradi")
vis = frame.copy()
vis[person.mask > 0] = (vis[person.mask > 0] * 0.4 + np.array([255, 0, 255]) * 0.6).astype(np.uint8)
cv2.circle(vis, person.pointer, 10, (0, 255, 0), -1)
show(vis, titles=["maschera posa + scheletro braccia (dilatata), indice in verde"], size=10)

Funziona sul frame intero perché il detector interno ritaglia la persona prima di segmentarla.
La maschera della sola posa perdeva un tratto del braccio teso: `PersonSegmenter` lo copre
disegnando le braccia come segmenti spessi tra spalla, gomito, polso e indice.
`refine=True` aggiunge DeepLab sul ritaglio.

## 4 · Geometria: lavagna e pixel occlusi

In [ ]:
quad_raw = detect_board(frame)
quad = detect_board(frame, person.mask)
print("senza maschera persona:", None if quad_raw is None else quad_raw.astype(int).tolist())
print("con maschera persona:  ", quad.astype(int).tolist())
bm = board_mask(frame.shape, quad)
occ = (bm > 0) & (person.mask > 0)
print(f"pixel di lavagna occlusi: {occ.sum()} su {(bm > 0).sum()} ({occ.sum() / (bm > 0).sum():.1%})")

H, size = rectify_homography(quad)
vis = frame.copy(); cv2.polylines(vis, [quad.astype(np.int32)], True, (0, 0, 255), 3)
vis[occ] = (vis[occ] * 0.4 + np.array([255, 0, 255]) * 0.6).astype(np.uint8)
show(vis, cv2.warpPerspective(frame, H, size), titles=["lavagna e occlusione", "lavagna rettificata"])

Le gambe sotto la lavagna deformano il contorno se non si toglie prima la persona.
Su un video la soluzione più semplice è cercare la lavagna sulla **mediana temporale** dei frame,
dove il docente sparisce (`video.median_frame`); in alternativa i 4 angoli si fissano a mano una volta per aula.

## 5 · Colore: pelle, ardesia, gesso

In [ ]:
arm = cv2.imread(str(ROOT / "data/calibration/sample_arm.png"), cv2.IMREAD_UNCHANGED)
arm_px = arm[arm[..., 3] > 200][:, :3]
to_lab = lambda px: cv2.cvtColor(px.reshape(-1, 1, 3).astype(np.uint8), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(float)

board_px = to_lab(frame[(bm > 0) & (person.mask == 0)])
L = board_px[:, 0]
groups = {"braccio": to_lab(arm_px),
          "ardesia": board_px[L < np.percentile(L, 50)],
          "gesso": board_px[L > np.percentile(L, 97)]}
for name, px in groups.items():
    print(f"{name:8s} Lab mediana {np.median(px, 0).round(0)}  std {px.std(0).round(1)}")

fig, ax = plt.subplots(figsize=(6, 5))
for name, px in groups.items():
    s = px[np.random.default_rng(0).choice(len(px), min(3000, len(px)), replace=False)]
    ax.scatter(s[:, 1] - 128, s[:, 0], s=2, alpha=0.4, label=name)
ax.set_xlabel("a (verde - rosso)"); ax.set_ylabel("L (luminosità, scala 0-255)"); ax.legend(); plt.show()

L'ardesia è compattissima, quindi «non è lavagna» si riconosce bene. Ma braccio e gesso hanno la stessa
luminosità e differiscono di poco nel colore; una camicia bianca è indistinguibile dal gesso.
Il colore da solo non basta: serve forma (i tratti di gesso sono sottili) o il modello della persona.
Utile come rifinitura dei bordi della maschera, non come criterio principale.

## Conclusione

Sul frame singolo: **PoseLandmarker + scheletro delle braccia** per la persona, **geometria** per la lavagna.
Il passo decisivo però è temporale: la lavagna è ferma e il docente si muove, quindi ogni pixel
coperto ora è stato visibile poco prima o lo sarà poco dopo: `02_ricostruzione_temporale.ipynb`.